# Dimensionality - running CAPS analysis

In [11]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az

import matplotlib.pyplot as plt

# set randon seed for reproducibility
random_seed = 42

In [12]:
# combine caps data with other information
df_caps = pd.read_csv('data/caps_total.csv')
df = pd.read_csv('data/scr_brain_group.csv')



In [16]:
df = pd.merge(df, df_caps[['sub_id','total_caps']], on = 'sub_id', how = 'left')
df.head()

,sub,Condition,Event.Nr,CDA.AmpSum,pe,scr,index,subject,trialNo,condition,coupling,amg,amg_vmpfc,sub_id,group,Gender,Age,n_zero_scr_x,n_zero_scr_y,total_caps
0,sub-189,CSplusUS1,1,0.2852,0.500000,0.2852,3036,sub-189,1,CSplusUS1,0.904762,0.476625,0.833333,sub-189,HC,2.0,26.0,15,15,0.0
1,sub-189,CSminus1,2,0.1033,-0.500000,0.1033,3037,sub-189,2,CSminus1,-0.380952,0.081692,0.428571,sub-189,HC,2.0,26.0,15,15,0.0
2,sub-189,CSplus1,3,0.0783,-0.750000,0.0783,3038,sub-189,3,CSplus1,0.571429,-0.219659,0.690476,sub-189,HC,2.0,26.0,15,15,0.0
3,sub-189,CSplusUS1,4,0.1772,0.686106,0.1772,3039,sub-189,4,CSplusUS1,0.619048,0.006618,0.880952,sub-189,HC,2.0,26.0,15,15,0.0
4,sub-189,CSminus1,5,0.0000,-0.250000,0.0000,3040,sub-189,5,CSminus1,0.833333,-0.188212,0.595238,sub-189,HC,2.0,26.0,15,15,0.0


In [21]:
df[df['group']=='VCC']['total_caps'].describe()

count    1725.000000
mean       14.680000
std        16.333164
min         0.000000
25%         0.000000
50%         7.000000
75%        27.000000
max        50.000000
Name: total_caps, dtype: float64

# Model

In [8]:

# Trauma-exposed only
te = df[df['group'].isin(['VCC','VPTSD'])].copy()
te['sub_idx'] = pd.Categorical(te['sub_id']).codes
n_subs = te['sub_idx'].nunique()
te['caps_z'] = (te['total_caps'] - te['total_caps'].mean()) / te['total_caps'].std()

# rank transformation

#te['caps_rank'] = te['total_caps'].rank()
#te['caps_rank_z'] = (te['caps_rank'] - te['caps_rank'].mean())/te['caps_rank'].std()
# swap caps_z → caps_rank_z, refit, check b_cxcaps still credible


pe        = te['pe'].values
coupling  = te['amg_vmpfc'].values      # primary circuit; rerun with 'coupling' for hippocampus
amg       = te['amg'].values
trialNo   = te['trialNo'].values
sub_idx   = te['sub_idx'].values
caps_z    = te['caps_z'].values

with pm.Model() as model_dim:
    b_coupling = pm.Normal('b_coupling', 0, 1)   # slope at mean severity
    b_caps     = pm.Normal('b_caps', 0, 1)       # main effect of severity on PE
    b_cxcaps   = pm.Normal('b_cxcaps', 0, 1)     # KEY: does slope change with severity?
    b_amg      = pm.Normal('b_amg', 0, 1)
    b_trial    = pm.Normal('b_trial', 0, 1)
    mu_a = pm.Normal('mu_a', 0, 1); sigma_a = pm.HalfNormal('sigma_a', 1)
    z_a = pm.Normal('z_a', 0, 1, shape=n_subs)
    a = pm.Deterministic('a', mu_a + z_a*sigma_a)
    mu = (a[sub_idx] + b_coupling*coupling + b_caps*caps_z
          + b_cxcaps*(coupling*caps_z) + b_amg*amg + b_trial*trialNo)
    sigma = pm.HalfNormal('sigma', 1)
    pm.Normal('pe', mu=mu, sigma=sigma, observed=pe)
    trace_dim = pm.sample(chains=4, random_seed=42, return_inferencedata=True,
                          idata_kwargs={"log_likelihood": True})

az.summary(trace_dim, var_names=['b_coupling','b_caps','b_cxcaps'], hdi_prob=0.89)
# pd for the key interaction:
v = trace_dim.posterior['b_cxcaps'].values.ravel()
print(f'coupling×CAPS: mean={v.mean():+.3f}, 89%HDI={az.hdi(v,hdi_prob=0.89)}, pd={max((v>0).mean(),(v<0).mean())*100:.1f}%')

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [b_coupling, b_caps, b_cxcaps, b_amg, b_trial, mu_a, sigma_a, z_a, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 5 seconds.


coupling×CAPS: mean=+0.048, 89%HDI=[0.01633654 0.08067477], pd=99.3%


In [10]:
np.std(v)

np.float64(0.020156207560019485)

## Running it on amygdala-hippocampus now


In [20]:

# Trauma-exposed only
te = df[df['group'].isin(['VCC','VPTSD'])].copy()
te['sub_idx'] = pd.Categorical(te['sub_id']).codes
n_subs = te['sub_idx'].nunique()
te['caps_z'] = (te['total_caps'] - te['total_caps'].mean()) / te['total_caps'].std()

# rank transformation

te['caps_rank'] = te['total_caps'].rank()
te['caps_rank_z'] = (te['caps_rank'] - te['caps_rank'].mean())/te['caps_rank'].std()
# swap caps_z → caps_rank_z, refit, check b_cxcaps still credible


pe        = te['pe'].values
coupling  = te['coupling'].values      # primary circuit; rerun with 'coupling' for hippocampus
amg       = te['amg'].values
trialNo   = te['trialNo'].values
sub_idx   = te['sub_idx'].values
caps_z    = te['caps_rank_z'].values

with pm.Model() as model_dim:
    b_coupling = pm.Normal('b_coupling', 0, 1)   # slope at mean severity
    b_caps     = pm.Normal('b_caps', 0, 1)       # main effect of severity on PE
    b_cxcaps   = pm.Normal('b_cxcaps', 0, 1)     # KEY: does slope change with severity?
    b_amg      = pm.Normal('b_amg', 0, 1)
    b_trial    = pm.Normal('b_trial', 0, 1)
    mu_a = pm.Normal('mu_a', 0, 1); sigma_a = pm.HalfNormal('sigma_a', 1)
    z_a = pm.Normal('z_a', 0, 1, shape=n_subs)
    a = pm.Deterministic('a', mu_a + z_a*sigma_a)
    mu = (a[sub_idx] + b_coupling*coupling + b_caps*caps_z
          + b_cxcaps*(coupling*caps_z) + b_amg*amg + b_trial*trialNo)
    sigma = pm.HalfNormal('sigma', 1)
    pm.Normal('pe', mu=mu, sigma=sigma, observed=pe)
    trace_dim = pm.sample(chains=4, random_seed=42, return_inferencedata=True,
                          idata_kwargs={"log_likelihood": True})

az.summary(trace_dim, var_names=['b_coupling','b_caps','b_cxcaps'], hdi_prob=0.89)
# pd for the key interaction:
v = trace_dim.posterior['b_cxcaps'].values.ravel()
print(f'coupling×CAPS: mean={v.mean():+.3f}, 89%HDI={az.hdi(v,hdi_prob=0.89)}, pd={max((v>0).mean(),(v<0).mean())*100:.1f}%')

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [b_coupling, b_caps, b_cxcaps, b_amg, b_trial, mu_a, sigma_a, z_a, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 5 seconds.


coupling×CAPS: mean=+0.023, 89%HDI=[-0.02156193  0.06070252], pd=81.9%
